In [5]:
!pip install pgmpy

In [12]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# --------------------------
# 1️⃣ Define the Discrete Bayesian Network structure
# --------------------------
model = DiscreteBayesianNetwork([
    ('Rain', 'Sprinkler'),
    ('Rain', 'GrassWet'),
    ('Sprinkler', 'GrassWet')
])

# --------------------------
# 2️⃣ Define CPDs
# --------------------------

# Rain prior
cpd_rain = TabularCPD(
    variable='Rain',
    variable_card=2,
    values=[[0.8],  # Rain=0 (False)
            [0.2]]  # Rain=1 (True)
)

# Sprinkler | Rain
cpd_sprinkler = TabularCPD(
    variable='Sprinkler',
    variable_card=2,
    values=[
        [0.6, 0.99],  # Sprinkler=Off | Rain=0,1
        [0.4, 0.01]   # Sprinkler=On
    ],
    evidence=['Rain'],
    evidence_card=[2]
)

# GrassWet | Rain, Sprinkler (columns: Rain=0,Sprinkler=0; Rain=0,Sprinkler=1; ...)
cpd_grasswet = TabularCPD(
    variable='GrassWet',
    variable_card=2,
    values=[
        [0.9, 0.1, 0.1, 0.01],  # GrassWet=No
        [0.1, 0.9, 0.9, 0.99]   # GrassWet=Yes
    ],
    evidence=['Rain','Sprinkler'],  # correct order
    evidence_card=[2,2]
)

# --------------------------
# 3️⃣ Add CPDs
# --------------------------
model.add_cpds(cpd_rain, cpd_sprinkler, cpd_grasswet)
assert model.check_model()  # ensures all probabilities are valid

# --------------------------
# 4️⃣ Inference
# --------------------------
inference = VariableElimination(model)

# Posterior: Rain given GrassWet=True
result = inference.query(
    variables=['Rain'],
    evidence={'GrassWet':1}
)

print(result)

+---------+-------------+
| Rain    |   phi(Rain) |
+=========+=============+
| Rain(0) |      0.6509 |
+---------+-------------+
| Rain(1) |      0.3491 |
+---------+-------------+
